# 02 — Environment Full Pipeline Verification

Verify the complete simulation pipeline end-to-end:
1. Build resolved model with inventory
2. `init_state` — initial SimulationState
3. Empty phases — pure iteration
4. **DemandPhase** — demand reduces inventory, unmet demand logged
5. **ArrivalsPhase** — shipments arrive, resources freed
6. **DispatchPhase** — task dispatches, validation, auto-assign resources
7. Full pipeline: Demand + Arrivals + Dispatch(NoopTask)

In [ ]:
import dataclasses
import pandas as pd
from gbp.build.pipeline import build_model
from tests.unit.build.fixtures import minimal_raw_model

## 1. Build resolved model with inventory

In [ ]:
raw = minimal_raw_model(with_demand=True)
raw = dataclasses.replace(
    raw,
    inventory_initial=pd.DataFrame({
        "facility_id": ["d1", "s1", "s2"],
        "commodity_category": ["working_bike"] * 3,
        "quantity": [50.0, 12.0, 7.0],
        "quantity_unit": ["bike"] * 3,
    }),
)
resolved = build_model(raw)
print(f"Periods: {len(resolved.periods)}")
print(f"Facilities: {list(resolved.facilities['facility_id'])}")
print(f"\nDemand (resolved):")
display(resolved.demand)
print(f"\nEdges:")
display(resolved.edges[["source_id", "target_id", "modal_type", "lead_time_hours"]])

## 2. init_state

In [ ]:
from gbp.consumers.simulator.state import init_state

state = init_state(resolved)
print(f"Period: {state.period_id} (index={state.period_index})")
print(f"\nInventory:")
display(state.inventory)
print(f"\nResources:")
display(state.resources)

## 3. DemandPhase — demand consumes inventory

In [ ]:
from gbp.consumers.simulator.built_in_phases import DemandPhase, ArrivalsPhase
from gbp.consumers.simulator.state import PeriodRow

# Get period p0 as PeriodRow
periods = [PeriodRow(*row) for row in resolved.periods.itertuples()]
p0 = periods[0]
print(f"Period: {p0.period_id}, demand for this period:")
display(resolved.demand[resolved.demand["period_id"] == p0.period_id])

# Execute DemandPhase
demand_phase = DemandPhase()
result = demand_phase.execute(state, resolved, p0)

print(f"\nInventory after demand:")
display(result.state.inventory)
print(f"\nFlow events (commodity consumed):")
display(result.event("flow_events"))
print(f"\nUnmet demand:")
display(result.event("unmet_demand"))

## 4. ArrivalsPhase — shipments arrive

In [ ]:
# Manually add a shipment arriving at period 0 to test ArrivalsPhase
transit = pd.DataFrame({
    "shipment_id": ["shp_test"],
    "source_id": ["d1"],
    "target_id": ["s1"],
    "commodity_category": ["working_bike"],
    "quantity": [5.0],
    "resource_id": ["rebalancing_truck_d1_0"],
    "departure_period": [0],
    "arrival_period": [0],
})
state_with_transit = state.with_in_transit(transit)

# Also mark the resource as IN_TRANSIT
res = state_with_transit.resources.copy()
res.loc[res["resource_id"] == "rebalancing_truck_d1_0", "status"] = "in_transit"
state_with_transit = state_with_transit.with_resources(res)

arrivals_phase = ArrivalsPhase()
result = arrivals_phase.execute(state_with_transit, resolved, p0)

print("Inventory after arrivals (s1 should be 12 + 5 = 17):")
display(result.state.inventory)
print(f"\nIn-transit after arrivals: {len(result.state.in_transit)} shipments")
print(f"\nResource truck_d1_0 status:")
truck = result.state.resources[result.state.resources["resource_id"] == "rebalancing_truck_d1_0"]
display(truck[["resource_id", "status", "current_facility_id", "available_at_period"]])

## 5. DispatchPhase — task produces dispatches, validated and applied

In [ ]:
from gbp.consumers.simulator.dispatch_phase import DispatchPhase
from gbp.consumers.simulator.task import DISPATCH_COLUMNS

# Custom task: send 5 bikes from d1 to s1
class SendBikesTask:
    name = "send_bikes"
    def run(self, state, resolved, period):
        return pd.DataFrame({
            "source_id": ["d1"],
            "target_id": ["s1"],
            "commodity_category": ["working_bike"],
            "quantity": [5.0],
            "resource_id": [None],  # auto-assign
            "modal_type": ["road"],
            "arrival_period": [period.period_index + 1],
        })

dispatch_phase = DispatchPhase(task=SendBikesTask())
result = dispatch_phase.execute(state, resolved, p0)

print("Inventory after dispatch (d1 should be 50 - 5 = 45):")
display(result.state.inventory)
print(f"\nIn-transit ({len(result.state.in_transit)} shipments):")
display(result.state.in_transit)
print(f"\nResources (one truck should be IN_TRANSIT):")
display(result.state.resources[["resource_id", "status", "current_facility_id", "available_at_period"]])
print(f"\nFlow events:")
display(result.event("flow_events"))
print(f"\nRejected: {len(result.event('rejected_dispatches'))}")

## 6. Full pipeline: Demand + Arrivals + Dispatch(NoopTask) over 3 periods

In [ ]:
from gbp.consumers.simulator import Environment, EnvironmentConfig
from gbp.consumers.simulator.tasks.noop import NoopTask

config = EnvironmentConfig(
    phases=[
        DemandPhase(),
        ArrivalsPhase(),
        DispatchPhase(task=NoopTask()),
    ],
)
env = Environment(resolved, config)
log = env.run()
dfs = log.to_dataframes()

print("Log table shapes:")
for name, df in dfs.items():
    print(f"  {name}: {df.shape}")

In [ ]:
# Inventory evolution per period
inv = dfs["simulation_inventory_log"]
pivot = inv.pivot_table(index="period_id", columns="facility_id", values="quantity")
print("Inventory per period (demand reduces s1 and s2):")
display(pivot)
print(f"\nTotal per period:")
print(inv.groupby("period_id")["quantity"].sum())

In [ ]:
# Flow log
print("Flow events (demand consumption):")
display(dfs["simulation_flow_log"])

In [ ]:
# Unmet demand
print("Unmet demand:")
unmet = dfs["simulation_unmet_demand_log"]
if unmet.empty:
    print("  (none — all demand satisfied)")
else:
    display(unmet)